In [2]:
import pandas as pd

file_path = r"D:\Sushant\Documents\oncology_survival_project\data\seer\seer_analysis_dataset.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (1132076, 8)


,age,stage_clean,grade_clean,chemo,radiation,surgery,time,event
0,70,Localized,NaN,0,1,1,12.0,1
1,70,Localized,3.0,1,1,1,59.0,0
2,60,Regional,NaN,0,0,1,81.0,1
3,70,Regional,NaN,0,0,1,7.0,1
4,50,Regional,NaN,1,1,1,84.0,0


In [3]:
df = df.copy()

# rename for consistency
df = df.rename(columns={
    'time': 'survival_months'
})

# handle grade missing
df['grade_clean'] = df['grade_clean'].fillna(df['grade_clean'].median())

# drop any remaining missing
df = df.dropna()

print(df.shape)

(1132076, 8)


In [4]:
df['stage_clean'].value_counts()

stage_clean
Localized    738655
Regional     326413
Distant       67008
Name: count, dtype: int64

In [5]:
stage_map = {
    'Localized': 0,
    'Regional': 1,
    'Distant': 2
}

df['stage'] = df['stage_clean'].map(stage_map)

In [6]:
features = [
    'age',
    'stage',
    'grade_clean',
    'chemo',
    'radiation',
    'surgery'
]

X = df[features]
T = df['survival_months']
E = df['event']

In [7]:
from lifelines import CoxPHFitter

df_cox = df[features + ['survival_months', 'event']].copy()

cph = CoxPHFitter()
cph.fit(df_cox, duration_col='survival_months', event_col='event')

# 36-month survival probability
surv_fn = cph.predict_survival_function(df_cox, times=[36])

df['cox_prob'] = 1 - surv_fn.T.squeeze().values

In [11]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
import numpy as np

# --- STEP 1: Subsample for RSF ---
df_rsf = df.sample(n=100000, random_state=42)

X_rsf = df_rsf[features]
y_rsf = Surv.from_dataframe('event', 'survival_months', df_rsf)

# --- STEP 2: Train RSF ---
rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rsf.fit(X_rsf, y_rsf)

# --- STEP 3: Predict ONLY on subset ---
surv_probs = rsf.predict_survival_function(X_rsf, return_array=True)

time_grid = rsf.event_times_
t_idx = (abs(time_grid - 36)).argmin()

df_rsf['rsf_prob'] = 1 - surv_probs[:, t_idx]

AttributeError: 'RandomSurvivalForest' object has no attribute 'event_times_'